# 経費精算分析

このノートブックでは、ローカルの領収書画像から出張経費を処理し、経費精算メールを作成し、円グラフで経費データを可視化するプラグインを使用するエージェントの作り方を示します。エージェントはタスクの文脈に応じて動的に関数を選択します。

手順：
1. OCRエージェントがローカル領収書画像を処理し、出張経費データを抽出します。
2. メールエージェントが経費精算メールを生成します。

### 出張経費の例：
ある社員が別の都市でのビジネスミーティングのために出張していると想像してください。会社は合理的な出張関連経費をすべて払い戻す方針です。以下は潜在的な出張経費の内訳です：
- 交通費：
自宅の都市から目的地の都市への往復航空券。
空港までおよび空港からのタクシーまたはライドシェアサービス。
目的地都市での公共交通機関、レンタカー、タクシーなどの現地交通費。

- 宿泊費：
ミーティング会場近くの中程度のビジネスホテルに3泊。

- 食費：
会社の一日あたり支給規定に基づく朝食、昼食、夕食の食費。

- 雑費：
空港の駐車料金。
ホテルでのインターネット接続料金。
チップや小さなサービス料。

- 書類：
フライト、タクシー、ホテル、食事などの領収書すべてと、精算のために記入済みの経費報告書を提出します。


## 必要なライブラリのインポート

ノートブックに必要なライブラリとモジュールをインポートします。


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import base64
import dotenv
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

 ## 支出モデルの定義

 個々の支出用のPydanticモデルを作成し、ユーザーのクエリを構造化された支出データに変換するExpenseFormatterクラスを作成します。

 各支出は次の形式で表されます：
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## Defining Tools - Generating the Email

Create a tool function to generate an email for submitting an expense claim.
- This tool uses the `@tool` decorator from the Microsoft Agent Framework.
- It calculates the total amount of the expenses and formats the details into an email body.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# レシート画像から交通費を抽出するツール

レシート画像から交通費を抽出するツール関数を作成します。
- このツールは Microsoft Agent Framework の `@tool` デコレーターを使用します。
- レシート画像を読み込み、base64でエンコードし、エージェントが解析できるようにデータURIとして返します。


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## 処理経費

エージェントを定義し、`WorkflowBuilder` を使って順次ワークフローに接続します。
- OCR エージェントは、`load_receipt_image` ツールを使用して領収書画像から構造化された経費データを抽出します。
- Email エージェントは、抽出されたデータを受け取り、`generate_expense_email` ツールを使ってプロフェッショナルな経費請求メールを作成します。
- `add_edge` を用いた `WorkflowBuilder` により連続的なパイプラインを作成します：OCR エージェント → Email エージェント。


In [ ]:
ocr_agent = client.as_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = client.as_agent(
    name="EmailAgent",
    tools=[generate_expense_email],
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## メイン関数

領収書の画像を処理し、経費請求メールを生成するために、逐次ワークフローを構築して実行します。


> **注意:** このワークフローは現在、レシート画像をbase64テキストとして渡していますが、ほとんどのチャットモデル（gpt-5-miniを含む）はこれを画像として認識しません。
> また、モデルのコンテキストウィンドウを超える可能性もあります。Azure AI Vision（または他のOCRツール）でOCRを実行し、抽出したテキストのみを渡すか、画像を `image_url` メッセージとして送信するようにリファクタリングすることを推奨します。
> コンテキストエラーを回避したいだけの場合は、より小さいレシート画像を使用するか、より大きなコンテキストウィンドウを持つモデルを試してください。


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責事項**：
本書類は AI 翻訳サービス [Co-op Translator](https://github.com/Azure/co-op-translator) を使用して翻訳されています。正確性を期していますが、自動翻訳には誤りや不正確な部分が含まれる可能性があることをご承知おきください。原文の原語版が正式な情報源とみなされるべきです。重要な情報については、専門の人間による翻訳を推奨します。本翻訳の利用により生じたいかなる誤解や解釈違いについても、当方は責任を負いかねます。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
